This notebook trains a random forest model to predict the binning label of each image based on the features extracted from the cell profiling pipeline. 
The model is trained on a balanced dataset, where each binning label is represented equally. 
The dataset is split into training, validation, and test sets, and the model's performance is evaluated on the test set.
The predicted bins are used to adjust the segmentation parameters for each image, which is expected to improve the segmentation quality.

In [1]:
import argparse
import json
import os
import pathlib
import sys
import time

import joblib
import matplotlib.image as mpimg
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import psutil
import scipy
import tifffile
from arg_parsing_utils import check_for_missing_args, parse_args
from file_reading import *
from file_reading import read_zstack_image
from notebook_init_utils import bandicoot_check, init_notebook
from skimage.filters import sobel
from sklearn.ensemble import RandomForestClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import classification_report, confusion_matrix
from sklearn.model_selection import train_test_split

In [2]:
def read_labels(infile: str) -> dict:
    """
    Description
    ----------
    Read labels from a parquet file.
    Parameters
    ----------
    infile : str
        Path to the input parquet file.
    Returns
    -------
    dict
        Dictionary containing the labels.
    """
    data = pd.read_parquet(infile).to_dict(orient="list")
    return data

In [3]:
start_time = time.time()
# get starting memory (cpu)
start_mem = psutil.Process(os.getpid()).memory_info().rss / 1024**2

In [4]:
root_dir, in_notebook = init_notebook()

image_base_dir = bandicoot_check(
    pathlib.Path(os.path.expanduser("~/mnt/bandicoot/NF1_organoid_data")).resolve(),
    root_dir,
)
patient_list_file_path = pathlib.Path(f"{root_dir}/data/patient_IDs.txt").resolve(
    strict=True
)
raw_image_base_dir = pathlib.Path(f"{image_base_dir}/data/").resolve()

In [5]:
labels_input_file = pathlib.Path("../image_labels/annotations.parquet").resolve(
    strict=True
)
labels_save_file = pathlib.Path(
    "../image_labels/segmentation_classes.parquet"
).resolve()
labels_save_file.parent.mkdir(parents=True, exist_ok=True)
all_features_save_path = pathlib.Path(
    f"../../3.cellprofiling/results/all_features.parquet"
).resolve(strict=True)
labels = read_labels(labels_input_file)
labels_df = pd.DataFrame(labels)
labels_df["patient"] = labels_df["image_filename"].apply(
    lambda x: (
        "_".join(x.split("_")[0:2]) if not "CQ1" in x else "_".join(x.split("_")[0:3])
    )
)
labels_df["well_fov"] = labels_df["image_filename"].apply(
    lambda x: x.split("_")[2] if not "CQ1" in x else x.split("_")[3]
)
non_feature_cols = [
    "patient",
    "well_fov",
    "image_filename",
    "annotator",
    "label",
    "label_name",
    "timestamp",
]
all_features_df = pd.read_parquet(all_features_save_path)
all_features_df
df = pd.merge(
    all_features_df,
    labels_df,
    on=["patient", "well_fov"],
    how="left",
)
print(df.shape)

(3734, 3847)


In [6]:
log_reg_model_path = "../models/logistic_regression_model.joblib"
rf_model_path = "../models/random_forest_model.joblib"
log_reg_model = joblib.load(log_reg_model_path)
rf_model = joblib.load(rf_model_path)

In [7]:
predicted_labels = log_reg_model.predict(df.drop(columns=non_feature_cols))
prediction_df = df.assign(predicted_label=predicted_labels)
prediction_df.drop(
    columns=[col for col in prediction_df.columns if "CHAMI" in col or "SAM" in col],
    inplace=True,
)
prediction_df.drop(columns=["timestamp", "image_filename", "label"], inplace=True)
prediction_df.head()

,patient,well_fov,label_name,annotator,predicted_label
0,NF0014_T1,F11-2,dissociated,Mike,dissociated
1,NF0014_T1,F10-1,dissociated,Mike,dissociated
2,NF0014_T1,D6-1,globular,Mike,globular
3,NF0014_T1,G5-2,globular,Mike,globular
4,NF0014_T1,F6-1,globular,Mike,globular


In [8]:
output_dict = {"patient": [], "well_fov": [], "label": [], "predicted_or_gt": []}

for index, row in prediction_df.iterrows():
    output_dict["patient"].append(row["patient"])
    output_dict["well_fov"].append(row["well_fov"])
    if pd.isna(row["predicted_label"]):
        output_dict["label"].append(row["predicted_label"])
        output_dict["predicted_or_gt"].append("predicted")
    else:
        output_dict["label"].append(row["label_name"])
        output_dict["predicted_or_gt"].append("gt")

df = pd.DataFrame(output_dict)
df.to_parquet(labels_save_file, index=False)
print(df.shape)
df.head()

(3734, 4)


,patient,well_fov,label,predicted_or_gt
0,NF0014_T1,F11-2,dissociated,gt
1,NF0014_T1,F10-1,dissociated,gt
2,NF0014_T1,D6-1,globular,gt
3,NF0014_T1,G5-2,globular,gt
4,NF0014_T1,F6-1,globular,gt
